# Tests de mini-funcionalidades de OP-10 `write_flows`

Este notebook se usa para probar helpers y bloques internos de `write_flows()` antes de hacer smoke tests o tests integrados de la función pública completa.

Objetivo:

- verificar minifuncionalidades de escritura de forma aislada;
- detectar errores de implementación temprano;
- asegurar que el bloque de persistencia de flows respete el contrato backend-aware actual;
- dejar una base fácil de portar después a `pytest`.

Convenciones:

- los tests usan `assert`;
- se prueban helpers internos de OP-10, no la función pública completa;
- no se incluyen todavía smoke tests;
- no se prueban helpers de `read_flows`, porque esos corresponden a OP-11.

### 0.1 Imports generales

Qué prepara: imports básicos, utilidades de filesystem y dependencias Arrow necesarias para revisar escrituras Parquet/Feather.

In [1]:
import copy
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.feather as feather

### 0.2 Imports del módulo

Qué prepara: imports de clases y helpers reales de `pylondrina.io.flows` usados por OP-10 `write_flows`.

In [2]:
from pylondrina.datasets import FlowDataset
from pylondrina.reports import Issue
from pylondrina.errors import ExportError

from pylondrina.io.flows import (
    WriteFlowsOptions,

    _validate_write_contract,
    _freeze_flow_write_snapshot,

    _create_flows_staging_dir,
    _write_flows_table_to_staging,
    _write_optional_flow_to_trips_to_staging,
    _write_flow_sidecar_to_staging,
    _assert_flows_staging_complete,
    _commit_staged_flow_bundle,
    _cleanup_staging_dir,

    _normalize_flows_artifact_root_for_write,
    _resolve_flows_artifact_paths,

    _flow_data_filename_for_storage,
    _flow_to_trips_filename_for_storage,
    _build_flow_storage_options_snapshot,

    _build_write_flows_summary,
    _options_to_write_parameters,
    _build_issues_summary,
    _build_io_event,
    _append_event,

    _assert_json_safe,

    _collect_flow_arrow_categorical_fields,
    _prepare_flows_df_for_arrow_write,

    _ensure_dataset_id,
    _new_artifact_id,
)

### 0.3 Helpers de apoyo para test

Qué prepara: utilidades pequeñas para assertions, inspección de issues y lectura mínima de archivos escritos en Parquet/Feather.

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def assert_json_dumpable(obj, label: str = "object"):
    try:
        json.dumps(obj, ensure_ascii=False)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e


def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]


def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró el issue {code}. Codes actuales: {codes}"


def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró inesperadamente el issue {code}. Codes actuales: {codes}"


def assert_counts_by_level(issues, *, errors=None, warnings=None, info=None):
    counts = {"error": 0, "warning": 0, "info": 0}
    for issue in issues:
        counts[issue.level] = counts.get(issue.level, 0) + 1

    if errors is not None:
        assert counts["error"] == errors, (
            f"errors esperado={errors}, actual={counts['error']}"
        )
    if warnings is not None:
        assert counts["warning"] == warnings, (
            f"warnings esperado={warnings}, actual={counts['warning']}"
        )
    if info is not None:
        assert counts["info"] == info, (
            f"info esperado={info}, actual={counts['info']}"
        )


def read_written_table(path: Path, storage_format: str) -> pd.DataFrame:
    if storage_format == "parquet":
        return pd.read_parquet(path, engine="pyarrow")
    if storage_format == "feather":
        return feather.read_feather(path)
    raise AssertionError(f"storage_format inesperado: {storage_format!r}")

### 0.4 Configuración visual

Qué prepara: display más cómodo para inspección manual ocasional.

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## Bloque 1. Fixtures reutilizables mínimas

Qué prepara: factories pequeñas para construir `FlowDataset`, tablas auxiliares y sidecars sintéticos usados por los tests helper-level de OP-10.

In [5]:
HELPER_ROOT = Path("./tmp_helper_write_flows")


def reset_helper_root() -> Path:
    if HELPER_ROOT.exists():
        shutil.rmtree(HELPER_ROOT)
    HELPER_ROOT.mkdir(parents=True, exist_ok=True)
    return HELPER_ROOT


def make_case_dir(case_name: str) -> Path:
    case_dir = HELPER_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    return case_dir


def make_flows_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f2", "f3"],
            "origin_h3_index": [
                "881111111111111",
                "882222222222222",
                "883333333333333",
            ],
            "destination_h3_index": [
                "884444444444444",
                "885555555555555",
                "886666666666666",
            ],
            "flow_count": [2, 1, 3],
            "flow_value": [2.0, 1.0, 4.5],
            "mode": ["bus", "metro", "bus"],
            "window_start_utc": [
                "2026-01-01T08:00:00Z",
                "2026-01-01T09:00:00Z",
                "2026-01-01T10:00:00Z",
            ],
            "window_end_utc": [
                "2026-01-01T08:59:59Z",
                "2026-01-01T09:59:59Z",
                "2026-01-01T10:59:59Z",
            ],
        }
    )


def make_flow_to_trips_df() -> pd.DataFrame:
    return pd.DataFrame(
        {
            "flow_id": ["f1", "f1", "f2"],
            "movement_id": ["m1", "m2", "m3"],
        }
    )


def make_flowdataset(
    *,
    validated: bool = False,
    include_dataset_id: bool = True,
    include_artifact_id: bool = False,
    include_aux: bool = True,
    aggregation_spec: dict | None = None,
    metadata_extra: dict | None = None,
    provenance: dict | None = None,
) -> FlowDataset:
    metadata = {
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-06T00:00:00Z",
                "parameters": {"h3_resolution": 8},
                "summary": {"n_flows": 3},
                "issues_summary": {
                    "counts": {"info": 0, "warning": 0, "error": 0},
                    "top_codes": [],
                },
            }
        ],
        "is_validated": validated,
    }

    if include_dataset_id:
        metadata["dataset_id"] = "dset_existing"

    if include_artifact_id:
        metadata["artifact_id"] = "art_existing"

    if metadata_extra:
        metadata.update(copy.deepcopy(metadata_extra))

    if aggregation_spec is None:
        aggregation_spec = {
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
        }

    if provenance is None:
        provenance = {
            "derived_from": [
                {"type": "trips", "dataset_id": "trip_dset_001"},
            ],
            "prior_events_summary": {"build_flows": 1},
        }

    return FlowDataset(
        flows=make_flows_df(),
        flow_to_trips=make_flow_to_trips_df() if include_aux else None,
        aggregation_spec=copy.deepcopy(aggregation_spec),
        source_trips="SENTINEL_SOURCE_TRIPS_ONLY_IN_MEMORY",
        metadata=copy.deepcopy(metadata),
        provenance=copy.deepcopy(provenance),
    )


def make_sidecar_payload(
    *,
    storage_format: str = "feather",
    include_flow_to_trips: bool = True,
) -> dict:
    if storage_format == "parquet":
        data_name = "flows.parquet"
        aux_name = "flow_to_trips.parquet"
        storage_options = {"compression": "snappy"}
    elif storage_format == "feather":
        data_name = "flows.feather"
        aux_name = "flow_to_trips.feather"
        storage_options = {"compression": "lz4", "version": 2}
    else:
        raise ValueError(f"storage_format inesperado: {storage_format!r}")

    return {
        "dataset_type": "flows",
        "format": "golondrina",
        "layout_version": "1.1",
        "storage": {
            "format": storage_format,
            "options": storage_options,
        },
        "dataset_id": "dset_sidecar",
        "artifact_id": "art_sidecar",
        "files": {
            "data": data_name,
            "metadata": "flows.metadata.json",
            "flow_to_trips": aux_name if include_flow_to_trips else None,
        },
        "aggregation_spec": {
            "h3_resolution": 8,
            "group_by": ["mode"],
            "time_aggregation": "hour",
            "time_basis": "origin",
            "min_trips_per_flow": 1,
        },
        "provenance": {
            "derived_from": [
                {"type": "trips", "dataset_id": "trip_dset_001"}
            ],
        },
        "metadata": {
            "dataset_id": "dset_sidecar",
            "artifact_id": "art_sidecar",
            "is_validated": False,
            "events": [],
        },
        "tables": {
            "flows": {
                "n_rows": 3,
                "n_cols": len(make_flows_df().columns),
                "columns": list(make_flows_df().columns),
            },
            "flow_to_trips": {
                "n_rows": 3,
                "n_cols": len(make_flow_to_trips_df().columns),
                "columns": list(make_flow_to_trips_df().columns),
            } if include_flow_to_trips else None,
        },
    }


root = reset_helper_root()
print("HELPER_ROOT =", root.resolve())
show_ok("Bloque 1 - fixtures mínimas")

HELPER_ROOT = C:\projects\pylondrina\notebooks\testing\io_flows\tmp_helper_write_flows
OK - Bloque 1 - fixtures mínimas


## Bloque 2. Helpers de layout, nombres físicos y evidencia

Este bloque prueba helpers transversales usados por OP-10 antes de materializar archivos:

- normalización del root `.golondrina`;
- resolución de rutas base;
- nombres físicos dependientes del backend;
- snapshot de opciones de almacenamiento;
- parameters serializables;
- JSON-safety;
- identidad y evidencia mínima de eventos.

### Test 2.1 - `_normalize_flows_artifact_root_for_write`

Qué prueba:

- agrega el sufijo `.golondrina` cuando corresponde;
- no duplica el sufijo si ya está presente;
- respeta `normalize_artifact_dir=False`.

In [6]:
p1 = _normalize_flows_artifact_root_for_write(
    "demo_flows",
    normalize_artifact_dir=True,
)
assert p1.name == "demo_flows.golondrina"

p2 = _normalize_flows_artifact_root_for_write(
    "demo_flows.golondrina",
    normalize_artifact_dir=True,
)
assert p2.name == "demo_flows.golondrina"

p3 = _normalize_flows_artifact_root_for_write(
    "demo_flows",
    normalize_artifact_dir=False,
)
assert p3.name == "demo_flows"

show_ok("Test 2.1 - _normalize_flows_artifact_root_for_write")

OK - Test 2.1 - _normalize_flows_artifact_root_for_write


### Test 2.2 - `_resolve_flows_artifact_paths`

Qué prueba:

- el helper solo resuelve el root del bundle y el sidecar;
- los nombres de tablas ya no viven fijos en `FlowsArtifactPaths`, porque dependen del backend físico.

In [7]:
root = Path("tmp_demo_flows_artifact")
paths = _resolve_flows_artifact_paths(root)

assert paths.root_dir == root
assert paths.sidecar_path == root / "flows.metadata.json"

# Esta ausencia es intencional en la versión backend-aware.
assert not hasattr(paths, "data_path")
assert not hasattr(paths, "flow_to_trips_path")

show_ok("Test 2.2 - _resolve_flows_artifact_paths")

OK - Test 2.2 - _resolve_flows_artifact_paths


### Test 2.3 - nombres de archivos según backend

Qué prueba:

- Parquet produce `flows.parquet` y `flow_to_trips.parquet`;
- Feather produce `flows.feather` y `flow_to_trips.feather`;
- un backend no soportado falla temprano.

In [8]:
assert _flow_data_filename_for_storage("parquet") == "flows.parquet"
assert _flow_to_trips_filename_for_storage("parquet") == "flow_to_trips.parquet"

assert _flow_data_filename_for_storage("feather") == "flows.feather"
assert _flow_to_trips_filename_for_storage("feather") == "flow_to_trips.feather"

try:
    _flow_data_filename_for_storage("csv")
    raise AssertionError("Debió fallar por backend no soportado")
except ValueError:
    pass

try:
    _flow_to_trips_filename_for_storage("csv")
    raise AssertionError("Debió fallar por backend no soportado")
except ValueError:
    pass

show_ok("Test 2.3 - nombres físicos por backend")

OK - Test 2.3 - nombres físicos por backend


### Test 2.4 - `_build_flow_storage_options_snapshot`

Qué prueba:

- Parquet serializa solo su compresión;
- Feather serializa compresión y `version=2`;
- el sidecar puede distinguir correctamente ambos backends.

In [9]:
parquet_options = WriteFlowsOptions(
    storage_format="parquet",
    parquet_compression="zstd",
)
parquet_storage = _build_flow_storage_options_snapshot(parquet_options)

assert parquet_storage == {"compression": "zstd"}


feather_options = WriteFlowsOptions(
    storage_format="feather",
    feather_compression="lz4",
)
feather_storage = _build_flow_storage_options_snapshot(feather_options)

assert feather_storage == {"compression": "lz4", "version": 2}

show_ok("Test 2.4 - _build_flow_storage_options_snapshot")

OK - Test 2.4 - _build_flow_storage_options_snapshot


### Test 2.5 - `_options_to_write_parameters`

Qué prueba:

- los parámetros efectivos de write quedan serializables;
- el bloque preserva backend, compresiones, política de overwrite y decisión sobre `flow_to_trips`.

In [12]:
options = WriteFlowsOptions(
    mode="overwrite",
    storage_format="feather",
    parquet_compression="snappy",
    feather_compression="zstd",
    normalize_artifact_dir=True,
    write_flow_to_trips=False,
)

parameters = _options_to_write_parameters(
    path=Path("/tmp/demo_flows.golondrina"),
    options=options,
)

assert parameters["path"] == str(Path("/tmp/demo_flows.golondrina"))
assert parameters["mode"] == "overwrite"
assert parameters["storage_format"] == "feather"
assert parameters["parquet_compression"] == "snappy"
assert parameters["feather_compression"] == "zstd"
assert parameters["normalize_artifact_dir"] is True
assert parameters["write_flow_to_trips"] is False

assert_json_dumpable(parameters, "write_parameters")
show_ok("Test 2.5 - _options_to_write_parameters")

OK - Test 2.5 - _options_to_write_parameters


### Test 2.6 - `_assert_json_safe`

Qué prueba:

- acepta un payload JSON-safe;
- falla con issue estructurado cuando el bloque no es serializable.

In [13]:
issues_ok = []

_assert_json_safe(
    {"a": 1, "b": ["x", "y"]},
    label="payload_ok",
    issues=issues_ok,
    path=Path("/tmp/fake"),
    artifact="flows.metadata.json",
)

assert issues_ok == []


issues_bad = []

try:
    _assert_json_safe(
        {"bad": {1, 2, 3}},
        label="payload_bad",
        issues=issues_bad,
        path=Path("/tmp/fake"),
        artifact="flows.metadata.json",
    )
    raise AssertionError("Debió fallar por payload no serializable")
except ExportError:
    assert_issue_present(
        issues_bad,
        "WRITE_FLOWS.SNAPSHOT.NOT_JSON_SERIALIZABLE",
    )

show_ok("Test 2.6 - _assert_json_safe")

OK - Test 2.6 - _assert_json_safe


### Test 2.7 - `_ensure_dataset_id` y `_new_artifact_id`

Qué prueba:

- preservación de `dataset_id` válido;
- creación cuando falta;
- regeneración cuando existe pero es inválido;
- generación de `artifact_id` nuevo para la materialización.

In [14]:
dataset_id, status = _ensure_dataset_id({"dataset_id": "dset_ok"})
assert dataset_id == "dset_ok"
assert status == "preserved"


dataset_id2, status2 = _ensure_dataset_id({})
assert isinstance(dataset_id2, str)
assert dataset_id2.startswith("dset_")
assert status2 == "created"


dataset_id3, status3 = _ensure_dataset_id({"dataset_id": ""})
assert isinstance(dataset_id3, str)
assert dataset_id3.startswith("dset_")
assert status3 == "regenerated"


artifact_id = _new_artifact_id()
assert isinstance(artifact_id, str)
assert artifact_id.startswith("art_")

show_ok("Test 2.7 - dataset_id / artifact_id")

OK - Test 2.7 - dataset_id / artifact_id


### Test 2.8 - `_build_issues_summary`, `_build_io_event` y `_append_event`

Qué prueba:

- resumen compacto de issues por severidad y código;
- forma mínima del evento IO;
- append-only del evento sin mutar la metadata original.

In [15]:
issues = [
    Issue(
        level="warning",
        code="WRITE_FLOWS.TEST.WARNING_A",
        message="warning A",
    ),
    Issue(
        level="warning",
        code="WRITE_FLOWS.TEST.WARNING_A",
        message="warning A repeated",
    ),
    Issue(
        level="info",
        code="WRITE_FLOWS.TEST.INFO_B",
        message="info B",
    ),
]

issues_summary = _build_issues_summary(issues)

assert issues_summary["counts"] == {
    "info": 1,
    "warning": 2,
    "error": 0,
}
assert issues_summary["top_codes"][0] == {
    "code": "WRITE_FLOWS.TEST.WARNING_A",
    "count": 2,
}

event = _build_io_event(
    op="write_flows",
    parameters={"storage_format": "feather"},
    summary={"n_flows": 3},
    issues_summary=issues_summary,
)

assert event["op"] == "write_flows"
assert "ts_utc" in event
assert event["parameters"]["storage_format"] == "feather"
assert event["summary"]["n_flows"] == 3
assert event["issues_summary"]["counts"]["warning"] == 2

metadata = {
    "events": [{"op": "build_flows"}],
    "x": 1,
}

metadata_out = _append_event(metadata, event)

assert metadata is not metadata_out
assert len(metadata["events"]) == 1
assert len(metadata_out["events"]) == 2
assert metadata_out["events"][-1]["op"] == "write_flows"
assert metadata_out["x"] == 1

show_ok("Test 2.8 - issues summary / IO event / append event")

OK - Test 2.8 - issues summary / IO event / append event


## Bloque 3. Preparación Arrow compartida

La implementación actual ya no trata la preparación categórica como una optimización exclusiva de Parquet. Antes de escribir tanto Parquet como Feather, OP-10:

- detecta campos de segmentación relevantes desde `aggregation_spec["group_by"]`;
- excluye identificadores y columnas estructurales;
- convierte los campos aplicables a `pandas.Categorical`;
- remueve categorías no usadas.

Este bloque prueba esas dos mini-funcionalidades de forma aislada.

### Test 3.1 - `_collect_flow_arrow_categorical_fields`

Qué prueba:

- detecta campos de segmentación textuales;
- ignora columnas estructurales excluidas;
- ignora groupings numéricos o inexistentes;
- acepta `group_by` como lista o string.

In [16]:
df = make_flows_df()

categorical_fields = _collect_flow_arrow_categorical_fields(
    df,
    aggregation_spec={
        "group_by": [
            "mode",          # sí corresponde
            "flow_id",       # excluido
            "flow_count",    # numérico
            "missing_field", # no existe
        ]
    },
)

assert categorical_fields == ["mode"]


categorical_fields_str = _collect_flow_arrow_categorical_fields(
    df,
    aggregation_spec={"group_by": "mode"},
)

assert categorical_fields_str == ["mode"]

show_ok("Test 3.1 - _collect_flow_arrow_categorical_fields")

OK - Test 3.1 - _collect_flow_arrow_categorical_fields


### Test 3.2 - `_prepare_flows_df_for_arrow_write`

Qué prueba:

- hace copia defensiva del dataframe;
- convierte columnas objetivo a tipo categórico;
- remueve categorías no usadas;
- no muta el dataframe original.

In [17]:
df = make_flows_df()
df["mode"] = pd.Categorical(
    df["mode"],
    categories=["bus", "metro", "train"],
)

prepared = _prepare_flows_df_for_arrow_write(
    df,
    categorical_fields=["mode"],
)

assert prepared is not df
assert isinstance(prepared["mode"].dtype, pd.CategoricalDtype)
assert list(prepared["mode"].cat.categories) == ["bus", "metro"]

# El input original no se muta.
assert list(df["mode"].cat.categories) == ["bus", "metro", "train"]


df_plain = make_flows_df()
prepared_plain = _prepare_flows_df_for_arrow_write(
    df_plain,
    categorical_fields=["mode"],
)

assert isinstance(prepared_plain["mode"].dtype, pd.CategoricalDtype)
assert list(df_plain["mode"]) == ["bus", "metro", "bus"]

show_ok("Test 3.2 - _prepare_flows_df_for_arrow_write")

OK - Test 3.2 - _prepare_flows_df_for_arrow_write


## Bloque 4. Contrato de write y snapshot serializable

Este bloque prueba los helpers que fijan la semántica de OP-10 antes de tocar disco:

- validación fatal de precondiciones;
- soporte de Parquet y Feather;
- validación de compresiones;
- congelamiento del snapshot serializable;
- decisión sobre `flow_to_trips`;
- construcción del summary estable.

### Test 4.1 - `_validate_write_contract` happy path

Qué prueba:

- el contrato de escritura acepta correctamente tanto Feather como Parquet;
- no emite issues en escenarios válidos.

In [18]:
flows = make_flowdataset()

issues_feather = []
_validate_write_contract(
    flows,
    Path("/tmp/fake_feather.golondrina"),
    WriteFlowsOptions(storage_format="feather"),
    issues=issues_feather,
)
assert issues_feather == []


issues_parquet = []
_validate_write_contract(
    flows,
    Path("/tmp/fake_parquet.golondrina"),
    WriteFlowsOptions(
        storage_format="parquet",
        parquet_compression="snappy",
    ),
    issues=issues_parquet,
)
assert issues_parquet == []

show_ok("Test 4.1 - _validate_write_contract happy path")

OK - Test 4.1 - _validate_write_contract happy path


### Test 4.2 - `_validate_write_contract` fatal por dataset inválido

Qué prueba: precondición básica de que OP-10 requiere un `FlowDataset` usable.

In [19]:
issues = []

try:
    _validate_write_contract(
        object(),
        Path("/tmp/fake_artifact.golondrina"),
        WriteFlowsOptions(),
        issues=issues,
    )
    raise AssertionError("Debió fallar por dataset inválido")
except ExportError:
    assert_issue_present(
        issues,
        "WRITE_FLOWS.INPUT.INVALID_DATASET",
    )

show_ok("Test 4.2 - invalid FlowDataset")

OK - Test 4.2 - invalid FlowDataset


### Test 4.3 - `_validate_write_contract` fatal por `mode` inválido

Qué prueba: la política de colisión del bundle debe ser interpretable y pertenecer al contrato cerrado de OP-10.

In [20]:
flows = make_flowdataset()
issues = []

try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_artifact.golondrina"),
        WriteFlowsOptions(mode="append"),  # valor no soportado
        issues=issues,
    )
    raise AssertionError("Debió fallar por mode inválido")
except ExportError:
    assert_issue_present(
        issues,
        "WRITE_FLOWS.OPTIONS.INVALID_MODE",
    )

show_ok("Test 4.3 - invalid mode")

OK - Test 4.3 - invalid mode


### Test 4.4 - `_validate_write_contract` backend y compresiones inválidas

Qué prueba:

- falla si `storage_format` no es soportado;
- falla si la compresión Parquet no pertenece al conjunto permitido;
- falla si la compresión Feather no pertenece al conjunto permitido.

In [21]:
flows = make_flowdataset()

# Backend inválido
issues_backend = []
try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_invalid_backend.golondrina"),
        WriteFlowsOptions(storage_format="orc"),
        issues=issues_backend,
    )
    raise AssertionError("Debió fallar por backend no soportado")
except ExportError:
    assert_issue_present(
        issues_backend,
        "WRITE_FLOWS.OPTIONS.UNSUPPORTED_STORAGE_FORMAT",
    )


# Compresión Parquet inválida
issues_parquet_comp = []
try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_invalid_parquet_comp.golondrina"),
        WriteFlowsOptions(
            storage_format="parquet",
            parquet_compression="xz",
        ),
        issues=issues_parquet_comp,
    )
    raise AssertionError("Debió fallar por compresión Parquet inválida")
except ExportError:
    assert_issue_present(
        issues_parquet_comp,
        "WRITE_FLOWS.OPTIONS.UNSUPPORTED_STORAGE_FORMAT",
    )


# Compresión Feather inválida
issues_feather_comp = []
try:
    _validate_write_contract(
        flows,
        Path("/tmp/fake_invalid_feather_comp.golondrina"),
        WriteFlowsOptions(
            storage_format="feather",
            feather_compression="gzip",
        ),
        issues=issues_feather_comp,
    )
    raise AssertionError("Debió fallar por compresión Feather inválida")
except ExportError:
    assert_issue_present(
        issues_feather_comp,
        "WRITE_FLOWS.OPTIONS.UNSUPPORTED_STORAGE_FORMAT",
    )

show_ok("Test 4.4 - unsupported backend / invalid compressions")

OK - Test 4.4 - unsupported backend / invalid compressions


### Test 4.5 - `_freeze_flow_write_snapshot` con backend Feather y auxiliar presente

Qué prueba:

- Feather es el backend efectivo por defecto;
- se preserva `dataset_id` existente;
- se genera siempre un `artifact_id` nuevo;
- `flow_to_trips` entra al bundle cuando existe y fue solicitado;
- el sidecar resuelve nombres físicos Feather;
- el evento `write_flows` queda incorporado antes del write;
- `source_trips` no se serializa.

In [22]:
flows = make_flowdataset(
    include_dataset_id=True,
    include_artifact_id=True,
    include_aux=True,
)

paths = _resolve_flows_artifact_paths(
    Path("/tmp/fake_snapshot_feather.golondrina")
)

snapshot = _freeze_flow_write_snapshot(
    flows,
    paths,
    WriteFlowsOptions(),
    existing_issues=[],
)

assert snapshot.dataset_id_status == "preserved"
assert snapshot.dataset_id == "dset_existing"

assert snapshot.artifact_id.startswith("art_")
assert snapshot.artifact_id != "art_existing"

assert snapshot.files_written == [
    "flows.feather",
    "flows.metadata.json",
    "flow_to_trips.feather",
]
assert snapshot.n_flow_to_trips == 3

assert snapshot.sidecar_payload["storage"]["format"] == "feather"
assert snapshot.sidecar_payload["storage"]["options"] == {
    "compression": "lz4",
    "version": 2,
}

assert snapshot.sidecar_payload["files"]["data"] == "flows.feather"
assert snapshot.sidecar_payload["files"]["metadata"] == "flows.metadata.json"
assert snapshot.sidecar_payload["files"]["flow_to_trips"] == "flow_to_trips.feather"

assert snapshot.sidecar_payload["tables"]["flows"]["n_rows"] == 3
assert snapshot.sidecar_payload["tables"]["flow_to_trips"]["n_rows"] == 3

assert snapshot.metadata_for_persist["dataset_id"] == "dset_existing"
assert snapshot.metadata_for_persist["artifact_id"] == snapshot.artifact_id
assert snapshot.metadata_for_persist["events"][-1]["op"] == "write_flows"

assert "source_trips" not in json.dumps(
    snapshot.sidecar_payload,
    ensure_ascii=False,
)

assert snapshot.issues == []
assert_json_dumpable(snapshot.sidecar_payload, "snapshot_feather_sidecar")

show_ok("Test 4.5 - _freeze_flow_write_snapshot feather")

OK - Test 4.5 - _freeze_flow_write_snapshot feather


### Test 4.6 - `_freeze_flow_write_snapshot` con Parquet, `dataset_id` faltante y auxiliar ausente

Qué prueba:

- se crea `dataset_id` cuando falta;
- el sidecar se adapta a Parquet;
- si `write_flow_to_trips=True` pero no existe auxiliar, se deja evidencia;
- el bundle no declara archivo auxiliar inexistente.

In [23]:
flows = make_flowdataset(
    include_dataset_id=False,
    include_artifact_id=False,
    include_aux=False,
    validated=False,
)

paths = _resolve_flows_artifact_paths(
    Path("/tmp/fake_snapshot_parquet.golondrina")
)

snapshot = _freeze_flow_write_snapshot(
    flows,
    paths,
    WriteFlowsOptions(
        storage_format="parquet",
        parquet_compression="snappy",
        write_flow_to_trips=True,
    ),
    existing_issues=[],
)

assert snapshot.dataset_id_status == "created"
assert snapshot.dataset_id.startswith("dset_")
assert snapshot.artifact_id.startswith("art_")

assert snapshot.files_written == [
    "flows.parquet",
    "flows.metadata.json",
]
assert snapshot.n_flow_to_trips is None

assert snapshot.sidecar_payload["storage"]["format"] == "parquet"
assert snapshot.sidecar_payload["storage"]["options"] == {
    "compression": "snappy",
}

assert snapshot.sidecar_payload["files"]["data"] == "flows.parquet"
assert snapshot.sidecar_payload["files"]["flow_to_trips"] is None

assert_issue_present(
    snapshot.issues,
    "WRITE_FLOWS.METADATA.DATASET_ID_CREATED",
)
assert_issue_present(
    snapshot.issues,
    "WRITE_FLOWS.FLOW_TO_TRIPS.REQUESTED_BUT_MISSING",
)

assert_json_dumpable(snapshot.sidecar_payload, "snapshot_parquet_sidecar")
show_ok("Test 4.6 - _freeze_flow_write_snapshot parquet + missing aux")

OK - Test 4.6 - _freeze_flow_write_snapshot parquet + missing aux


### Test 4.7 - `_freeze_flow_write_snapshot` fatal por `aggregation_spec` inválido

Qué prueba: `aggregation_spec` es parte estructural del sidecar de flows y no puede ser una forma no interpretable.

In [24]:
flows = make_flowdataset()
flows.aggregation_spec = None

paths = _resolve_flows_artifact_paths(
    Path("/tmp/fake_invalid_aggregation_spec.golondrina")
)

try:
    _freeze_flow_write_snapshot(
        flows,
        paths,
        WriteFlowsOptions(),
        existing_issues=[],
    )
    raise AssertionError("Debió fallar por aggregation_spec inválido")
except ExportError as exc:
    # La evidencia se construye internamente antes del abort.
    assert isinstance(exc, ExportError)

show_ok("Test 4.7 - invalid aggregation_spec")

OK - Test 4.7 - invalid aggregation_spec


### Test 4.8 - `_build_write_flows_summary`

Qué prueba: summary mínimo estable de la operación de escritura.

In [25]:
summary = _build_write_flows_summary(
    n_flows=3,
    n_flow_to_trips=7,
    path=Path("/tmp/artifact.golondrina"),
    dataset_id="dset_001",
    artifact_id="art_001",
    files_written=[
        "flows.feather",
        "flows.metadata.json",
        "flow_to_trips.feather",
    ],
)

assert summary["n_flows"] == 3
assert summary["n_flow_to_trips"] == 7
assert Path(summary["path"]) == Path("/tmp/artifact.golondrina")
assert summary["dataset_id"] == "dset_001"
assert summary["artifact_id"] == "art_001"
assert summary["files_written"] == [
    "flows.feather",
    "flows.metadata.json",
    "flow_to_trips.feather",
]

assert_json_dumpable(summary, "write_flows_summary")
show_ok("Test 4.8 - _build_write_flows_summary")

OK - Test 4.8 - _build_write_flows_summary


## Bloque 5. Materialización en staging, sidecar y commit

Este bloque prueba los helpers que efectivamente materializan el bundle formal:

- creación y limpieza de staging;
- escritura real de tablas en Parquet y Feather;
- escritura opcional de `flow_to_trips`;
- escritura de `flows.metadata.json`;
- verificación de completitud mínima;
- commit final con política de colisión.

### Test 5.1 - `_create_flows_staging_dir` y `_cleanup_staging_dir`

Qué prueba:

- el staging se crea como directorio hermano del destino final;
- el cleanup remueve staging residual de manera normal.

In [26]:
with tempfile.TemporaryDirectory() as td:
    final_dir = Path(td) / "artifact_final.golondrina"
    issues = []

    staging_dir = _create_flows_staging_dir(
        final_dir,
        issues=issues,
    )

    assert staging_dir.exists()
    assert staging_dir.is_dir()
    assert staging_dir.parent == final_dir.parent
    assert issues == []

    _cleanup_staging_dir(
        staging_dir,
        final_dir,
        ["flows.feather", "flows.metadata.json"],
        issues,
    )

    assert not staging_dir.exists()

show_ok("Test 5.1 - staging create + cleanup")

OK - Test 5.1 - staging create + cleanup


### Test 5.2 - `_write_flows_table_to_staging` en Parquet y Feather

Qué prueba:

- la tabla principal se escribe correctamente en ambos backends;
- los archivos físicos esperados aparecen;
- el contenido puede leerse de vuelta;
- no se generan issues en el happy path.

In [27]:
case_dir = make_case_dir("case_05_02_write_flows_table")
staging = case_dir / "staging"
staging.mkdir(parents=True, exist_ok=True)

for storage_format, filename in [
    ("parquet", "flows.parquet"),
    ("feather", "flows.feather"),
]:
    data_path = staging / filename
    issues = []

    _write_flows_table_to_staging(
        make_flows_df(),
        data_path,
        storage_format=storage_format,
        parquet_compression="snappy",
        feather_compression="lz4",
        aggregation_spec={"group_by": ["mode"]},
        issues=issues,
        destination_path=case_dir,
    )

    assert data_path.exists()
    assert issues == []

    loaded = read_written_table(data_path, storage_format)
    assert list(loaded.columns) == list(make_flows_df().columns)
    assert len(loaded) == len(make_flows_df())

show_ok("Test 5.2 - write main flows table parquet/feather")

OK - Test 5.2 - write main flows table parquet/feather


### Test 5.3 - `_write_optional_flow_to_trips_to_staging`

Qué prueba:

- el auxiliar se escribe en Parquet y Feather cuando existe y fue solicitado;
- se omite limpiamente si el auxiliar es `None`;
- se omite limpiamente si `write_flow_to_trips=False`.

In [28]:
case_dir = make_case_dir("case_05_03_write_aux")
staging = case_dir / "staging"
staging.mkdir(parents=True, exist_ok=True)

for storage_format, filename in [
    ("parquet", "flow_to_trips.parquet"),
    ("feather", "flow_to_trips.feather"),
]:
    aux_path = staging / filename
    issues = []

    _write_optional_flow_to_trips_to_staging(
        make_flow_to_trips_df(),
        aux_path,
        write_flow_to_trips=True,
        storage_format=storage_format,
        parquet_compression="snappy",
        feather_compression="lz4",
        issues=issues,
        destination_path=case_dir,
    )

    assert aux_path.exists()
    assert issues == []

    loaded = read_written_table(aux_path, storage_format)
    assert list(loaded.columns) == ["flow_id", "movement_id"]
    assert len(loaded) == 3


# Omisión limpia por auxiliar ausente
aux_missing = staging / "flow_to_trips_missing.feather"
issues_missing = []

_write_optional_flow_to_trips_to_staging(
    None,
    aux_missing,
    write_flow_to_trips=True,
    storage_format="feather",
    parquet_compression="snappy",
    feather_compression="lz4",
    issues=issues_missing,
    destination_path=case_dir,
)

assert not aux_missing.exists()
assert issues_missing == []


# Omisión limpia porque el usuario no pidió persistirlo
aux_disabled = staging / "flow_to_trips_disabled.parquet"
issues_disabled = []

_write_optional_flow_to_trips_to_staging(
    make_flow_to_trips_df(),
    aux_disabled,
    write_flow_to_trips=False,
    storage_format="parquet",
    parquet_compression="snappy",
    feather_compression="lz4",
    issues=issues_disabled,
    destination_path=case_dir,
)

assert not aux_disabled.exists()
assert issues_disabled == []

show_ok("Test 5.3 - optional flow_to_trips write")

OK - Test 5.3 - optional flow_to_trips write


### Test 5.4 - `_write_flow_sidecar_to_staging`

Qué prueba:

- `flows.metadata.json` se materializa correctamente;
- el contenido escrito conserva backend, archivos y metadata;
- el helper no emite issues en el happy path.

In [29]:
case_dir = make_case_dir("case_05_04_sidecar_write")
staging = case_dir / "staging"
staging.mkdir(parents=True, exist_ok=True)

sidecar_path = staging / "flows.metadata.json"
payload = make_sidecar_payload(
    storage_format="feather",
    include_flow_to_trips=True,
)

issues = []

_write_flow_sidecar_to_staging(
    payload,
    sidecar_path,
    issues=issues,
    destination_path=case_dir,
)

assert sidecar_path.exists()
assert issues == []

loaded = json.loads(sidecar_path.read_text(encoding="utf-8"))

assert loaded["dataset_type"] == "flows"
assert loaded["storage"]["format"] == "feather"
assert loaded["storage"]["options"]["version"] == 2
assert loaded["files"]["data"] == "flows.feather"
assert loaded["files"]["flow_to_trips"] == "flow_to_trips.feather"
assert loaded["metadata"]["dataset_id"] == "dset_sidecar"

show_ok("Test 5.4 - _write_flow_sidecar_to_staging")

OK - Test 5.4 - _write_flow_sidecar_to_staging


### Test 5.5 - `_assert_flows_staging_complete`

Qué prueba:

- acepta un staging completo para un bundle Feather con auxiliar;
- falla si falta un archivo esperado.

In [30]:
# Caso completo
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_ok"
    staging_root.mkdir()

    paths = _resolve_flows_artifact_paths(staging_root)

    (staging_root / "flows.feather").touch()
    (staging_root / "flows.metadata.json").touch()
    (staging_root / "flow_to_trips.feather").touch()

    issues = []

    _assert_flows_staging_complete(
        paths,
        expected_files=[
            "flows.feather",
            "flows.metadata.json",
            "flow_to_trips.feather",
        ],
        issues=issues,
        destination_path=staging_root,
    )

    assert issues == []


# Caso incompleto
with tempfile.TemporaryDirectory() as td:
    staging_root = Path(td) / "staging_bad"
    staging_root.mkdir()

    paths = _resolve_flows_artifact_paths(staging_root)

    (staging_root / "flows.feather").touch()
    (staging_root / "flows.metadata.json").touch()
    # Falta flow_to_trips.feather

    issues = []

    try:
        _assert_flows_staging_complete(
            paths,
            expected_files=[
                "flows.feather",
                "flows.metadata.json",
                "flow_to_trips.feather",
            ],
            issues=issues,
            destination_path=staging_root,
        )
        raise AssertionError("Debió fallar por staging incompleto")
    except ExportError:
        assert_issue_present(
            issues,
            "WRITE_FLOWS.IO.STAGING_INCOMPLETE",
        )

show_ok("Test 5.5 - _assert_flows_staging_complete")

OK - Test 5.5 - _assert_flows_staging_complete


### Test 5.6 - `_commit_staged_flow_bundle` success

Qué prueba:

- el staging completo se promueve al directorio final;
- staging desaparece;
- no se generan issues en el happy path.

In [31]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact_final.golondrina"
    staging = parent / "staging"

    staging.mkdir()
    (staging / "flows.feather").write_text("dummy data", encoding="utf-8")
    (staging / "flows.metadata.json").write_text("{}", encoding="utf-8")

    issues = []

    _commit_staged_flow_bundle(
        staging,
        final_dir,
        mode="error_if_exists",
        files_written=[
            "flows.feather",
            "flows.metadata.json",
        ],
        issues=issues,
    )

    assert final_dir.exists()
    assert (final_dir / "flows.feather").exists()
    assert (final_dir / "flows.metadata.json").exists()
    assert not staging.exists()
    assert issues == []

show_ok("Test 5.6 - commit success")

OK - Test 5.6 - commit success


### Test 5.7 - `_commit_staged_flow_bundle` con `overwrite`

Qué prueba:

- un bundle existente se reemplaza bajo política explícita `overwrite`;
- se emite issue informativo/observable de overwrite;
- el contenido previo deja de existir.

In [32]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact_final.golondrina"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old_file.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "flows.parquet").write_text("dummy parquet", encoding="utf-8")
    (staging / "flows.metadata.json").write_text("{}", encoding="utf-8")

    issues = []

    _commit_staged_flow_bundle(
        staging,
        final_dir,
        mode="overwrite",
        files_written=[
            "flows.parquet",
            "flows.metadata.json",
        ],
        issues=issues,
    )

    assert final_dir.exists()
    assert (final_dir / "flows.parquet").exists()
    assert (final_dir / "flows.metadata.json").exists()
    assert not (final_dir / "old_file.txt").exists()
    assert not staging.exists()

    assert_issue_present(
        issues,
        "WRITE_FLOWS.LAYOUT.BUNDLE_OVERWRITTEN",
    )

show_ok("Test 5.7 - commit overwrite")

OK - Test 5.7 - commit overwrite


### Test 5.8 - `_commit_staged_flow_bundle` fatal por colisión

Qué prueba: `mode="error_if_exists"` no permite sobrescribir silenciosamente un bundle existente.

In [33]:
with tempfile.TemporaryDirectory() as td:
    parent = Path(td)
    final_dir = parent / "artifact_final.golondrina"
    staging = parent / "staging"

    final_dir.mkdir()
    (final_dir / "old_file.txt").write_text("legacy", encoding="utf-8")

    staging.mkdir()
    (staging / "flows.feather").write_text("dummy feather", encoding="utf-8")
    (staging / "flows.metadata.json").write_text("{}", encoding="utf-8")

    issues = []

    try:
        _commit_staged_flow_bundle(
            staging,
            final_dir,
            mode="error_if_exists",
            files_written=[
                "flows.feather",
                "flows.metadata.json",
            ],
            issues=issues,
        )
        raise AssertionError("Debió fallar por destino ya existente")
    except ExportError:
        assert_issue_present(
            issues,
            "WRITE_FLOWS.LAYOUT.BUNDLE_EXISTS",
        )

show_ok("Test 5.8 - commit collision")

OK - Test 5.8 - commit collision
